In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "knowledge_base").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from my_rag import search as s

anchors = s._load_anchors()

def check(text):
    q = s._encode(text)
    a = s._anchor_vecs @ q
    c = s._vectors @ q
    diff = float(a.max() - c.max())
    results = s.search(text)
    top = results[0]["source"] if results else "BLOCKED"
    print(round(diff, 3), "|", top, "|", text)
    if not results:
        print("        anchor:", anchors[int(a.argmax())])

TESTS = [
    "How long does it take to travel from the hotel to the pyramids?",
    "How long does it take to get to the airport from the hotel?",
    "How far is Tahrir Square from the hotel?",
    "Is the hotel close to the Egyptian Museum?",
    "How long is the taxi ride to Khan el-Khalili?",
    "Combien de temps pour aller aux pyramides depuis l'hôtel ?",
    "Сколько километров от отеля до Хан эль-Халили?",
    "كم تستغرق الرحلة من الفندق إلى الأهرامات؟",
    "من عندكم للأهرامات بتاخد وقت قد إيه بالعربية؟",
]

OUTSIDE = [
    "Which pyramids should I visit and how much are the tickets?",
    "What are the opening hours of the Egyptian Museum?",
    "Recommend a good restaurant in Zamalek",
]

print("SHOULD PASS:")
for t in TESTS:
    check(t)
print("\nSHOULD BE BLOCKED:")
for t in OUTSIDE:
    check(t)

/Users/sondosahmed/projects/HotelAI-Intelligent-Agent_Final_Data/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1102 09:31:32.270000 99547 .venv/lib/python3.12/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Python(99882) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 44814.12it/s]


SHOULD PASS:
0.062 | location_and_surroundings.txt | How long does it take to travel from the hotel to the pyramids?
-0.106 | location_and_surroundings.txt | How long does it take to get to the airport from the hotel?
-0.115 | location_and_surroundings.txt | How far is Tahrir Square from the hotel?
-0.104 | location_and_surroundings.txt | Is the hotel close to the Egyptian Museum?
-0.029 | location_and_surroundings.txt | How long is the taxi ride to Khan el-Khalili?
0.031 | location_and_surroundings.txt | Combien de temps pour aller aux pyramides depuis l'hôtel ?
-0.026 | location_and_surroundings.txt | Сколько километров от отеля до Хан эль-Халили?
-0.128 | location_and_surroundings.txt | كم تستغرق الرحلة من الفندق إلى الأهرامات؟
-0.027 | location_and_surroundings.txt | من عندكم للأهرامات بتاخد وقت قد إيه بالعربية؟

SHOULD BE BLOCKED:
0.433 | BLOCKED | Which pyramids should I visit and how much are the tickets?
        anchor: Which pyramids should I visit and how much are the tickets

In [2]:
import csv
from my_rag.config import FAQ_FILE

NEW = [
    ("FAQ3930", "How far are the Giza Pyramids from the hotel?"),
    ("FAQ3930", "How many minutes is the drive from the hotel to the Pyramids of Giza?"),
    ("FAQ3930", "How far is the Grand Egyptian Museum from the hotel?"),
    ("FAQ3932", "À quelle distance de l'hôtel se trouvent les pyramides de Gizeh ?"),
    ("FAQ3932", "Combien de minutes en voiture jusqu'au Grand Musée égyptien ?"),
    ("FAQ3933", "Как далеко от отеля пирамиды Гизы?"),
]

with open(FAQ_FILE, encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    fields = reader.fieldnames
    rows = list(reader)

by_id = {}
questions = set()
max_id = 0
for r in rows:
    by_id[r["faq_id"]] = r
    questions.add(r["question"])
    max_id = max(max_id, int(r["faq_id"].replace("FAQ", "")))

added = []
for base_id, question in NEW:
    if question in questions:
        continue
    max_id += 1
    row = dict(by_id[base_id])
    row["faq_id"] = f"FAQ{max_id}"
    row["question"] = question
    row["split"] = "train"
    added.append(row)

with open(FAQ_FILE, "rb") as f:
    f.seek(-1, 2)
    last = f.read(1)

with open(FAQ_FILE, "a", encoding="utf-8", newline="") as f:
    if last != b"\n":
        f.write("\r\n")
    writer = csv.DictWriter(f, fieldnames=fields)
    for row in added:
        writer.writerow(row)

print(len(rows), "->", len(rows) + len(added))

4009 -> 4009
